多日的价格，看看能够组成成本线  
A股散户平均持股天数（2025–2026最新）  
权威口径（中登/交易所）  
- 2025年一季度（中登）：7.3天（极端高频）  
- 2025全年（上交所/换手率推算）：18天（年换手率≈600%，按250交易日）  
- 历史/长期均值（深交所）：32天  
- 2026年3月（投保基金）：不足9天（最新高频）
按资金规模分化（最常见）  
- <10万（占比超90%）：5–10天（超短线）  
- 10–50万：10–20天（短线  
- >100万（高净值）：30–60天（偏中长线）  
对比机构  
- 公募/机构平均持股：120–190天  
- 散户换手率是机构的5–8倍  


In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4938,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-06      0  6.45  6.58  6.33   6.49  8.520715e+08   
            2018-07-09      1  6.51  6.76  6.50   6.76  1.255008e+09   
            2018-07-10      2  6.76  6.76  6.66   6.73  8.030390e+08   
            2018-07-11      3  6.56  6.61  6.50   6.58  7.447658e+08   
            2018-07-12      4  6.54  6.82  6.52   6.75  1.008659e+09   

                             volume  
code        date                     
000001.XSHE 2018-07-06  131957580.0  
            2018-07-09  188260099.0  
            2018-07-10  119750900.0  
            2018-07-11  113666923.0  
            2018-07-12  149984167.0

In [3]:
DAYS = 8 # 暂时以8天为例子
dt[f'volume{DAYS}_sum']=dt['volume'].rolling(window=DAYS).sum()
dt.tail()

index   open   high    low  close         money  \
code        date                                                          
000001.XSHE 2026-09-23   1995  11.43  11.46  11.30  11.35  1.052015e+09   
            2026-09-24   1996  11.35  11.47  11.29  11.30  1.186737e+09   
            2026-09-28   1997  11.28  11.41  11.27  11.30  8.105758e+08   
            2026-09-29   1998  11.30  11.41  11.28  11.35  7.846328e+08   
            2026-09-30   1999  11.36  11.65  11.33  11.57  1.205815e+09   

                             volume  volume8_sum  
code        date                                  
000001.XSHE 2026-09-23   92669758.0  661744456.0  
            2026-09-24  104381872.0  690368821.0  
            2026-09-28   71534143.0  684714269.0  
            2026-09-29   69097909.0  656757858.0  
            2026-09-30  104535745.0  690577003.0

In [4]:
dt[f'cost{DAYS}']=(dt['close'] * (dt['volume']/dt[f'volume{DAYS}_sum'])).rolling(window=DAYS).sum()
dt.tail()

index   open   high    low  close         money  \
code        date                                                          
000001.XSHE 2026-09-23   1995  11.43  11.46  11.30  11.35  1.052015e+09   
            2026-09-24   1996  11.35  11.47  11.29  11.30  1.186737e+09   
            2026-09-28   1997  11.28  11.41  11.27  11.30  8.105758e+08   
            2026-09-29   1998  11.30  11.41  11.28  11.35  7.846328e+08   
            2026-09-30   1999  11.36  11.65  11.33  11.57  1.205815e+09   

                             volume  volume8_sum      cost8  
code        date                                             
000001.XSHE 2026-09-23   92669758.0  661744456.0  11.512066  
            2026-09-24  104381872.0  690368821.0  11.951782  
            2026-09-28   71534143.0  684714269.0  11.771452  
            2026-09-29   69097909.0  656757858.0  11.307159  
            2026-09-30  104535745.0  690577003.0  11.782743

In [9]:
dt.index = dt.index.droplevel('code')
dt.head()

,index,open,high,low,close,money,volume,volume8_sum,cost8
date,,,,,,,,,
2018-07-06,0,6.45,6.58,6.33,6.49,8.520715e+08,131957580.0,NaN,NaN
2018-07-09,1,6.51,6.76,6.50,6.76,1.255008e+09,188260099.0,NaN,NaN
2018-07-10,2,6.76,6.76,6.66,6.73,8.030390e+08,119750900.0,NaN,NaN
2018-07-11,3,6.56,6.61,6.50,6.58,7.447658e+08,113666923.0,NaN,NaN
2018-07-12,4,6.54,6.82,6.52,6.75,1.008659e+09,149984167.0,NaN,NaN


In [19]:
# 然后这里画图
# 创建附加绘图
apds = [
    mpf.make_addplot(dt['cost8'], color='blue', width=1),
]
mpf.plot(dt, type='candle', addplot=apds, volume=True,warn_too_much_data=5000,figsize=(4200/96, 720/96), savefig='成本线.png')

In [20]:
dt.head(20)

,index,open,high,low,close,money,volume,volume8_sum,cost8
date,,,,,,,,,
2018-07-06,0,6.45,6.58,6.33,6.49,8.520715e+08,131957580.0,NaN,NaN
2018-07-09,1,6.51,6.76,6.50,6.76,1.255008e+09,188260099.0,NaN,NaN
2018-07-10,2,6.76,6.76,6.66,6.73,8.030390e+08,119750900.0,NaN,NaN
2018-07-11,3,6.56,6.61,6.50,6.58,7.447658e+08,113666923.0,NaN,NaN
2018-07-12,4,6.54,6.82,6.52,6.75,1.008659e+09,149984167.0,NaN,NaN
2018-07-13,5,6.78,6.80,6.71,6.75,5.354012e+08,79349223.0,NaN,NaN
2018-07-16,6,6.73,6.77,6.61,6.64,6.034277e+08,90720397.0,NaN,NaN
2018-07-17,7,6.65,6.65,6.59,6.63,3.263970e+08,49362461.0,9.230518e+08,NaN
2018-07-18,8,6.65,6.73,6.61,6.62,4.606974e+08,69061931.0,8.601561e+08,NaN


In [21]:
# 这里拆开计算吧
dt[f'volume{DAYS}_percent'] = dt['volume'] / dt[f'volume{DAYS}_sum']
dt.head(20)

,index,open,high,low,close,money,volume,volume8_sum,cost8,volume8_percent
date,,,,,,,,,,
2018-07-06,0,6.45,6.58,6.33,6.49,8.520715e+08,131957580.0,NaN,NaN,NaN
2018-07-09,1,6.51,6.76,6.50,6.76,1.255008e+09,188260099.0,NaN,NaN,NaN
2018-07-10,2,6.76,6.76,6.66,6.73,8.030390e+08,119750900.0,NaN,NaN,NaN
2018-07-11,3,6.56,6.61,6.50,6.58,7.447658e+08,113666923.0,NaN,NaN,NaN
2018-07-12,4,6.54,6.82,6.52,6.75,1.008659e+09,149984167.0,NaN,NaN,NaN
2018-07-13,5,6.78,6.80,6.71,6.75,5.354012e+08,79349223.0,NaN,NaN,NaN
2018-07-16,6,6.73,6.77,6.61,6.64,6.034277e+08,90720397.0,NaN,NaN,NaN
2018-07-17,7,6.65,6.65,6.59,6.63,3.263970e+08,49362461.0,9.230518e+08,NaN,0.053477
2018-07-18,8,6.65,6.73,6.61,6.62,4.606974e+08,69061931.0,8.601561e+08,NaN,0.080290


In [22]:
dt[f'close{DAYS}_percent'] = dt['close'] * dt[f'volume{DAYS}_percent'] # 当天收盘价乘以比例
dt.head(20)

,index,open,high,low,close,money,volume,volume8_sum,cost8,volume8_percent,close8_percent
date,,,,,,,,,,,
2018-07-06,0,6.45,6.58,6.33,6.49,8.520715e+08,131957580.0,NaN,NaN,NaN,NaN
2018-07-09,1,6.51,6.76,6.50,6.76,1.255008e+09,188260099.0,NaN,NaN,NaN,NaN
2018-07-10,2,6.76,6.76,6.66,6.73,8.030390e+08,119750900.0,NaN,NaN,NaN,NaN
2018-07-11,3,6.56,6.61,6.50,6.58,7.447658e+08,113666923.0,NaN,NaN,NaN,NaN
2018-07-12,4,6.54,6.82,6.52,6.75,1.008659e+09,149984167.0,NaN,NaN,NaN,NaN
2018-07-13,5,6.78,6.80,6.71,6.75,5.354012e+08,79349223.0,NaN,NaN,NaN,NaN
2018-07-16,6,6.73,6.77,6.61,6.64,6.034277e+08,90720397.0,NaN,NaN,NaN,NaN
2018-07-17,7,6.65,6.65,6.59,6.63,3.263970e+08,49362461.0,9.230518e+08,NaN,0.053477,0.354556
2018-07-18,8,6.65,6.73,6.61,6.62,4.606974e+08,69061931.0,8.601561e+08,NaN,0.080290,0.531520


In [24]:
dt[f'cost{DAYS}']=dt[f'close{DAYS}_percent'].rolling(window=DAYS).sum()
dt.head(20)

,index,open,high,low,close,money,volume,volume8_sum,cost8,volume8_percent,close8_percent
date,,,,,,,,,,,
2018-07-06,0,6.45,6.58,6.33,6.49,8.520715e+08,131957580.0,NaN,NaN,NaN,NaN
2018-07-09,1,6.51,6.76,6.50,6.76,1.255008e+09,188260099.0,NaN,NaN,NaN,NaN
2018-07-10,2,6.76,6.76,6.66,6.73,8.030390e+08,119750900.0,NaN,NaN,NaN,NaN
2018-07-11,3,6.56,6.61,6.50,6.58,7.447658e+08,113666923.0,NaN,NaN,NaN,NaN
2018-07-12,4,6.54,6.82,6.52,6.75,1.008659e+09,149984167.0,NaN,NaN,NaN,NaN
2018-07-13,5,6.78,6.80,6.71,6.75,5.354012e+08,79349223.0,NaN,NaN,NaN,NaN
2018-07-16,6,6.73,6.77,6.61,6.64,6.034277e+08,90720397.0,NaN,NaN,NaN,NaN
2018-07-17,7,6.65,6.65,6.59,6.63,3.263970e+08,49362461.0,9.230518e+08,NaN,0.053477,0.354556
2018-07-18,8,6.65,6.73,6.61,6.62,4.606974e+08,69061931.0,8.601561e+08,NaN,0.080290,0.531520


# 用另一种公式
$AMOV=VOL * (OPEN+CLOSE)/2$  
$AMV_{n} = \frac{\sum{AMOV}}{\sum{VOL}}$

In [25]:
dt = datasource.getData(code)
dt['amov']=dt['volume']*(dt['open']+dt['close'])/2
dt.index = dt.index.droplevel('code')
dt.head()

index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-06      0  6.45  6.58  6.33   6.49  8.520715e+08   
            2018-07-09      1  6.51  6.76  6.50   6.76  1.255008e+09   
            2018-07-10      2  6.76  6.76  6.66   6.73  8.030390e+08   
            2018-07-11      3  6.56  6.61  6.50   6.58  7.447658e+08   
            2018-07-12      4  6.54  6.82  6.52   6.75  1.008659e+09   

                             volume          amov  
code        date                                   
000001.XSHE 2018-07-06  131957580.0  8.537655e+08  
            2018-07-09  188260099.0  1.249106e+09  
            2018-07-10  119750900.0  8.077198e+08  
            2018-07-11  113666923.0  7.467917e+08  
            2018-07-12  149984167.0  9.966448e+08

In [26]:
DAYS = 5
dt[f'amv{DAYS}']=dt['amov'].rolling(window=DAYS).sum() / dt['volume'].rolling(window=DAYS).sum()
dt.head(10)

index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-06      0  6.45  6.58  6.33   6.49  8.520715e+08   
            2018-07-09      1  6.51  6.76  6.50   6.76  1.255008e+09   
            2018-07-10      2  6.76  6.76  6.66   6.73  8.030390e+08   
            2018-07-11      3  6.56  6.61  6.50   6.58  7.447658e+08   
            2018-07-12      4  6.54  6.82  6.52   6.75  1.008659e+09   
            2018-07-13      5  6.78  6.80  6.71   6.75  5.354012e+08   
            2018-07-16      6  6.73  6.77  6.61   6.64  6.034277e+08   
            2018-07-17      7  6.65  6.65  6.59   6.63  3.263970e+08   
            2018-07-18      8  6.65  6.73  6.61   6.62  4.606974e+08   
            2018-07-19      9  6.65  6.70  6.61   6.64  4.036743e+08   

                             volume          amov      amv5  
code        date                                             
000001.XSHE 2018-07-06  131957580.0  8.537655e+08       NaN  
            2018-07-09  188260099.0  1.249106e+09       NaN  
            2018-07-10  119750900.0  8.077198e+08       NaN  
            2018-07-11  113666923.0  7.467917e+08       NaN  
            2018-07-12  149984167.0  9.966448e+08  6.614408  
            2018-07-13   79349223.0  5.367975e+08  6.662034  
            2018-07-16   90720397.0  6.064659e+08  6.674994  
            2018-07-17   49362461.0  3.277667e+08  6.654064  
            2018-07-18   69061931.0  4.582259e+08  6.672854  
            2018-07-19   60674284.0  4.031806e+08  6.679978

In [28]:
apds = [
    mpf.make_addplot(dt['amv5'], color='blue', width=1),
]

mpf.plot(dt, type='candle', addplot=apds, volume=True,warn_too_much_data=5000,figsize=(4200/96, 720/96), savefig='成本线.png')

In [31]:
for d in [5,10,20,60,120]:
    dt[f'amv{d}']=dt['amov'].rolling(window=d).sum() / dt['volume'].rolling(window=d).sum()
apds = [
    mpf.make_addplot(dt['amv5'], color='blue', width=1),
    mpf.make_addplot(dt['amv10'], color='blue', width=1),
    mpf.make_addplot(dt['amv20'], color='blue', width=1),
    mpf.make_addplot(dt['amv60'], color='blue', width=1),
    mpf.make_addplot(dt['amv120'], color='blue', width=1),
]

mpf.plot(dt, type='candle', addplot=apds, volume=True,warn_too_much_data=5000,figsize=(4200/96, 720/96), savefig='成本线.png')

从结果上看，跟均线差不多